In [4]:
import numpy as np
import pandas as pd
import os
from keras.models import Model
from keras.layers import *
from tensorflow import keras
from tensorflow.keras.layers import Input
import tensorflow as tf
from keras.utils import plot_model
from tensorflow.keras.preprocessing.text import Tokenizer


In [5]:
tokenizer = Tokenizer()

In [6]:
with open("/content/next_word_predict.txt", "r", encoding="utf-8") as f:
    df = f.read()

In [7]:
tokenizer.fit_on_texts([df])

In [8]:
tokenizer.word_index

{'the': 1,
 'and': 2,
 'to': 3,
 'of': 4,
 'a': 5,
 'i': 6,
 '”': 7,
 'in': 8,
 'that': 9,
 'it': 10,
 'he': 11,
 'was': 12,
 'you': 13,
 'his': 14,
 'is': 15,
 'my': 16,
 'have': 17,
 'with': 18,
 'as': 19,
 'had': 20,
 'at': 21,
 'which': 22,
 'for': 23,
 'be': 24,
 'not': 25,
 'me': 26,
 'but': 27,
 'from': 28,
 'we': 29,
 'this': 30,
 'said': 31,
 'upon': 32,
 'there': 33,
 'holmes': 34,
 'him': 35,
 'so': 36,
 'her': 37,
 'she': 38,
 'all': 39,
 '’': 40,
 'been': 41,
 'your': 42,
 'on': 43,
 'very': 44,
 'by': 45,
 'one': 46,
 'are': 47,
 '“i': 48,
 'were': 49,
 'an': 50,
 'no': 51,
 'would': 52,
 'out': 53,
 'what': 54,
 'then': 55,
 'up': 56,
 'when': 57,
 'man': 58,
 'could': 59,
 'has': 60,
 'do': 61,
 'into': 62,
 'or': 63,
 'little': 64,
 'will': 65,
 'who': 66,
 'mr': 67,
 'if': 68,
 'some': 69,
 'down': 70,
 'see': 71,
 'now': 72,
 'our': 73,
 'should': 74,
 'may': 75,
 'am': 76,
 'us': 77,
 'over': 78,
 'they': 79,
 'can': 80,
 'more': 81,
 'think': 82,
 'about': 83,
 'mu

In [9]:
input_sequences = []
for sentence in df.split('\n'):
    tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]

    for i in range(1,len(tokenized_sentence)):
        input_sequences.append(tokenized_sentence[:i+1])


In [10]:
input_sequences

[[145, 4790],
 [145, 4790, 1],
 [145, 4790, 1, 1020],
 [145, 4790, 1, 1020, 4],
 [145, 4790, 1, 1020, 4, 128],
 [145, 4790, 1, 1020, 4, 128, 34],
 [145, 4790, 1, 1020, 4, 128, 34, 45],
 [145, 4790, 1, 1020, 4, 128, 34, 45, 611],
 [145, 4790, 1, 1020, 4, 128, 34, 45, 611, 2235],
 [145, 4790, 1, 1020, 4, 128, 34, 45, 611, 2235, 2236],
 [30, 1021],
 [30, 1021, 15],
 [30, 1021, 15, 23],
 [30, 1021, 15, 23, 1],
 [30, 1021, 15, 23, 1, 275],
 [30, 1021, 15, 23, 1, 275, 4],
 [30, 1021, 15, 23, 1, 275, 4, 394],
 [30, 1021, 15, 23, 1, 275, 4, 394, 2237],
 [30, 1021, 15, 23, 1, 275, 4, 394, 2237, 21],
 [30, 1021, 15, 23, 1, 275, 4, 394, 2237, 21, 51],
 [30, 1021, 15, 23, 1, 275, 4, 394, 2237, 21, 51, 1676],
 [30, 1021, 15, 23, 1, 275, 4, 394, 2237, 21, 51, 1676, 2],
 [30, 1021, 15, 23, 1, 275, 4, 394, 2237, 21, 51, 1676, 2, 18],
 [572, 51],
 [572, 51, 3398],
 [572, 51, 3398, 3399],
 [572, 51, 3398, 3399, 13],
 [572, 51, 3398, 3399, 13, 75],
 [572, 51, 3398, 3399, 13, 75, 817],
 [572, 51, 3398, 33

In [11]:
max_length = max([len(i) for i in input_sequences])
max_length

20

In [12]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
padded_input_sequences = pad_sequences(input_sequences, maxlen = max_length, padding='pre')
padded_input_sequences

array([[   0,    0,    0, ...,    0,  145, 4790],
       [   0,    0,    0, ...,  145, 4790,    1],
       [   0,    0,    0, ..., 4790,    1, 1020],
       ...,
       [   0,    0,    0, ...,    3,  360,   83],
       [   0,    0,    0, ...,  360,   83,  358],
       [   0,    0,    0, ...,   83,  358, 1673]], dtype=int32)

In [13]:
x = padded_input_sequences[:,:-1]
y = padded_input_sequences[:,-1]

In [14]:
x.shape

(101619, 19)

In [17]:
no_of_tokens = len(tokenizer.word_index) + 1
print(no_of_tokens)
# from tensorflow.keras.utils import to_categorical
# y = to_categorical(y,num_classes=no_of_tokens)
# print(y)

# This is not possible because it will take much more memory nearly 6GB

8932


In [16]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense


In [19]:
model = Sequential()
model.add(Embedding(no_of_tokens, output_dim=100, input_length=max_length-1))
model.add(LSTM(150))
# model.add(Dense(no_of_tokens, activation='relu'))
model.add(Dense(no_of_tokens, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [20]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [21]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [22]:
history = model.fit(x,y,epochs=50,batch_size=128,validation_split=0.1)

Epoch 1/50
715/715 ━━━━━━━━━━━━━━━━━━━━ 13s 10ms/step - accuracy: 0.0570 - loss: 6.4710 - val_accuracy: 0.0656 - val_loss: 6.7826
Epoch 2/50
715/715 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.0825 - loss: 5.9288 - val_accuracy: 0.0985 - val_loss: 6.5657
Epoch 3/50
715/715 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.1159 - loss: 5.5750 - val_accuracy: 0.1078 - val_loss: 6.4873
Epoch 4/50
715/715 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.1345 - loss: 5.3380 - val_accuracy: 0.1182 - val_loss: 6.4864
Epoch 5/50
715/715 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.1492 - loss: 5.1378 - val_accuracy: 0.1239 - val_loss: 6.4873
Epoch 6/50
715/715 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.1611 - loss: 4.9632 - val_accuracy: 0.1228 - val_loss: 6.5147
Epoch 7/50
715/715 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.1695 - loss: 4.8045 - val_accuracy: 0.1257 - val_loss: 6.5746
Epoch 8/50
715/715 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.1768 - loss: 4.6600 - val_accura

In [40]:
text = "The Adventure of the Blue"
token_text = tokenizer.texts_to_sequences([text])[0]
token_text = pad_sequences([token_text], maxlen=max_length-1, padding='pre')
predicted = np.argmax(model.predict(token_text))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step


In [35]:
# tokenizer.word_index.items()

In [41]:
for word,index in tokenizer.word_index.items():
  if index == predicted:
    print(word)
    break

carbuncle


In [42]:
# predict next word
def predict_word(text):
    token_text = tokenizer.texts_to_sequences([text])[0]
    token_text = pad_sequences([token_text], maxlen=max_length-1, padding='pre')
    predicted = np.argmax(model.predict(token_text))

    for word,index in tokenizer.word_index.items():
        if index == predicted:
            return word

predict_word("The Adventure of the Blue")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step


'carbuncle'

In [ ]:
# predict next 3 words
def predict_word(text):
    for i in range(1,4):
      token_text = tokenizer.texts_to_sequences([text])[0]
      token_text = pad_sequences([token_text], maxlen=max_length-1, padding='pre')
      predicted = np.argmax(model.predict(token_text))

      for word,index in tokenizer.word_index.items():
          if index == predicted:
              text = text + " " + word
              break
    return text
    
predict_word("The Adventures")